In [1]:
%pip install keras

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
fichier_json = open('modele/modele.json', 'r')
modele_json = fichier_json.read()
fichier_json.close()

from keras.models import model_from_json

modele = model_from_json(modele_json)
modele.load_weights("modele/modele.weights.h5")

classes = ["Un T-shirt/haut",
           "Un pantalon",
           "Un pull",
           "Une robe",
           "Un manteau",
           "Des sandales",
           "Une chemise",
           "Des baskets",
           "Un sac",
           "Une botte de cheville"]


In [7]:
import numpy as np
from PIL import Image, ImageFilter

def load_and_preprocess_image(image_path, target_size=28, content_size=20):
    """
    Load an image, convert it to grayscale, and preprocess for model input.
    Args:
        image_path: Path to the input file
        target_size: Size of the output image (default: 28x28)
        content_size: Size of the resized within the target (default: 20)
    Returns:
        numpy.ndarray: Preprocessed image array ready for model prediction
    """
    image = Image.open(image_path).convert('L')
    resized_img = resize_with_aspect_ratio(image, content_size)
    canvas = create_centered_image(resized_img, target_size)
    normalized_array = normalize_pixels(canvas)
    
    return normalized_array.reshape(1, target_size, target_size, 1)

def resize_with_aspect_ratio(image, target_dimension):
    """
    Resize image while maintaining aspect ratio.
    Args:
        image: PIL Image object
        target_dimension: Target size for the longest dimension
    Returns:
        PIL Image: Resized and sharpened image
    """
    width, height = image.size
    if width > height:
        new_width = target_dimension
        new_height = max(1, round(target_dimension * height / width))
    else:
        new_height = target_dimension
        new_width = max(1, round(target_dimension * width / height))
    return image.resize((new_width, new_height), Image.LANCZOS).filter(ImageFilter.SHARPEN)

def create_centered_image(image, canvas_size):
    """
    Place image in the center of a white canvas.
    
    Args:
        image: PIL Image to center
        canvas_size: Size of the square canvas
    Returns:
        PIL Image: Image centered on white canvas
    """
    canvas = Image.new('L', (canvas_size, canvas_size), 255)
    img_width, img_height = image.size
    x_offset = (canvas_size - img_width) // 2
    y_offset = (canvas_size - img_height) // 2
    canvas.paste(image, (x_offset, y_offset))
    return canvas

def normalize_pixels(image):
    pixels = np.array(image.getdata(), dtype=np.float32)
    return (255 - pixels) / 255.0

image_array = load_and_preprocess_image("images/pantalon.jpg")
probabilities = modele.predict(image_array)[0]
predicted_class = np.argmax(probabilities)
print(f"Selon moi l'image est : {classes[predicted_class]}\n")
for i, class_name in enumerate(classes):
    print(f"{class_name}: {probabilities[i] * 100:.2f}%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
Selon moi l'image est : Un pantalon

Un T-shirt/haut: 0.62%
Un pantalon: 97.33%
Un pull: 0.02%
Une robe: 0.17%
Un manteau: 0.00%
Des sandales: 0.00%
Une chemise: 0.01%
Des baskets: 0.00%
Un sac: 1.84%
Une botte de cheville: 0.00%


C:\Users\shawn\AppData\Local\Temp\ipykernel_18512\915445725.py:57: DeprecationWarning: Image.Image.getdata is deprecated and will be removed in Pillow 14 (2027-10-15). Use get_flattened_data instead.
  pixels = np.array(image.getdata(), dtype=np.float32)
